# Retail Customer Retention Intelligence
## Phase 1 — SQL Business Analytics

This phase uses PostgreSQL to analyze customer purchasing behavior and revenue patterns before predictive modeling.

The analysis focuses on:

- revenue trends over time;
- order and customer activity;
- repeat purchasing behavior;
- customer revenue concentration;
- purchase recency and retention signals.

The SQL queries use aggregations, Common Table Expressions (CTEs), joins, and window functions to answer business-oriented questions from the cleaned transaction data.

### 1.1 Database Connection

In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text


# Locate the project root and load database credentials.
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")


# Create the PostgreSQL connection engine.
DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database engine created successfully.")

Database engine created successfully.


In [2]:
# Verify the PostgreSQL connection and source table.
connection_check_query = """
SELECT
    COUNT(*) AS transaction_rows,
    COUNT(DISTINCT customer_id) AS customers,
    COUNT(DISTINCT invoice) AS invoices
FROM retail_transactions;
"""

with engine.connect() as connection:
    connection_check = pd.read_sql(
        text(connection_check_query),
        connection
    )

connection_check

,transaction_rows,customers,invoices
0,776614,5861,36629


### 1.2 Revenue Performance Over Time

Monthly revenue is analyzed to understand seasonality, growth patterns, and periods of unusually high or low sales activity.

In [3]:
monthly_revenue_query = """
SELECT
    DATE_TRUNC('month', invoice_date) AS month,
    ROUND(SUM(line_revenue), 2) AS revenue,
    COUNT(DISTINCT invoice) AS invoices,
    COUNT(DISTINCT customer_id) AS active_customers
FROM retail_transactions
GROUP BY DATE_TRUNC('month', invoice_date)
ORDER BY month;
"""

with engine.connect() as connection:
    monthly_revenue = pd.read_sql(
        text(monthly_revenue_query),
        connection
    )

monthly_revenue

,month,revenue,invoices,active_customers
0,2009-12-01,677916.07,1497,951
1,2010-01-01,537251.50,991,717
2,2010-02-01,497937.57,1093,771
3,2010-03-01,665973.67,1509,1051
4,2010-04-01,585127.43,1315,939
5,2010-05-01,592734.13,1365,965
6,2010-06-01,629540.45,1482,1035
7,2010-07-01,581487.96,1362,924
8,2010-08-01,594561.17,1276,910
9,2010-09-01,805544.81,1659,1136


#### Interpretation

Monthly revenue shows a clear seasonal pattern.

- Revenue rises sharply during **September–November** in both 2010 and 2011.
- **November is the strongest month** in both years, reaching about **£1.16M in 2010** and **£1.14M in 2011**.
- These peaks are accompanied by higher invoice counts and more active customers, suggesting that the increase is driven by broader customer activity rather than only a few unusually large purchases.
- Revenue falls substantially after the year-end peak and remains comparatively lower during the early months of each year.
- **December 2011 should not be compared directly with full months**, because the dataset ends on December 9, 2011.

Overall, the retailer exhibits strong year-end seasonality that should be considered later when analyzing retention and forecasting revenue.

#### 1.2.1 Month-over-Month Revenue Growth

Month-over-month growth is calculated using the SQL `LAG()` window function to compare each month's revenue with the previous month.

In [4]:
monthly_growth_query = """
WITH monthly_sales AS (
    SELECT
        DATE_TRUNC('month', invoice_date) AS month,
        SUM(line_revenue) AS revenue
    FROM retail_transactions
    GROUP BY DATE_TRUNC('month', invoice_date)
),

revenue_growth AS (
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY month) AS previous_month_revenue
    FROM monthly_sales
)

SELECT
    month,
    ROUND(revenue, 2) AS revenue,
    ROUND(previous_month_revenue, 2) AS previous_month_revenue,
    ROUND(
        100.0 * (revenue - previous_month_revenue)
        / NULLIF(previous_month_revenue, 0),
        2
    ) AS mom_growth_pct
FROM revenue_growth
ORDER BY month;
"""

with engine.connect() as connection:
    monthly_growth = pd.read_sql(
        text(monthly_growth_query),
        connection
    )

monthly_growth

,month,revenue,previous_month_revenue,mom_growth_pct
0,2009-12-01,677916.07,NaN,NaN
1,2010-01-01,537251.50,677916.07,-20.75
2,2010-02-01,497937.57,537251.50,-7.32
3,2010-03-01,665973.67,497937.57,33.75
4,2010-04-01,585127.43,665973.67,-12.14
5,2010-05-01,592734.13,585127.43,1.30
6,2010-06-01,629540.45,592734.13,6.21
7,2010-07-01,581487.96,629540.45,-7.63
8,2010-08-01,594561.17,581487.96,2.25
9,2010-09-01,805544.81,594561.17,35.49


#### Interpretation

Month-over-month growth reinforces the strong seasonal pattern observed in revenue.

- Revenue increased by **35.49% in September 2010**, followed by further growth in October and November.
- A similar year-end acceleration appears in 2011, with revenue rising **47.72% in September 2011**.
- The largest decline among complete months occurred in **December 2010 (-51.11%)**, immediately after the November peak.
- Other noticeable declines include **February 2011 (-21.40%)** and **April 2011 (-22.07%)**.
- The **December 2011 decline (-54.93%) should not be interpreted as a true month-over-month decline**, because the dataset contains only the first nine days of that month.

The repeated September-to-November acceleration provides further evidence of strong seasonal purchasing behavior.

#### 1.2.2 Monthly Average Order Value

Average order value (AOV) measures the average merchandise revenue generated per invoice and helps distinguish revenue growth caused by higher transaction volume from growth caused by larger purchases.

In [5]:
monthly_aov_query = """
SELECT
    DATE_TRUNC('month', invoice_date) AS month,
    ROUND(SUM(line_revenue), 2) AS revenue,
    COUNT(DISTINCT invoice) AS invoices,
    ROUND(
        SUM(line_revenue) / NULLIF(COUNT(DISTINCT invoice), 0),
        2
    ) AS average_order_value
FROM retail_transactions
GROUP BY DATE_TRUNC('month', invoice_date)
ORDER BY month;
"""

with engine.connect() as connection:
    monthly_aov = pd.read_sql(
        text(monthly_aov_query),
        connection
    )

monthly_aov

,month,revenue,invoices,average_order_value
0,2009-12-01,677916.07,1497,452.85
1,2010-01-01,537251.50,991,542.13
2,2010-02-01,497937.57,1093,455.57
3,2010-03-01,665973.67,1509,441.33
4,2010-04-01,585127.43,1315,444.96
5,2010-05-01,592734.13,1365,434.24
6,2010-06-01,629540.45,1482,424.79
7,2010-07-01,581487.96,1362,426.94
8,2010-08-01,594561.17,1276,465.96
9,2010-09-01,805544.81,1659,485.56


#### Interpretation

The year-end revenue peaks are primarily driven by increased order volume rather than unusually high order values.

- In **November 2010**, revenue reached about **£1.16M** with **2,570 invoices**, while average order value was only **£449.80**.
- In **November 2011**, revenue reached about **£1.14M** with **2,642 invoices**, while average order value was **£430.18**.
- September and October 2011 combined both higher order volume and relatively strong average order values, contributing to the rapid revenue increase before November.
- Average order value varies substantially across months, but the largest revenue months generally correspond to the highest invoice volumes.
- The **December 2011 AOV of £660.09 should be interpreted cautiously**, since only the first nine days of December are available.

Overall, seasonal revenue growth appears to be driven mainly by a larger number of purchases, with order value providing a secondary contribution.

### 1.3 Customer and Order Activity

Customer activity is analyzed to understand how frequently customers purchase and how much repeat purchasing contributes to the retailer's transaction volume.

#### 1.3.1 Customer Purchase Frequency

The number of distinct invoices per customer is calculated to distinguish one-time buyers from repeat customers.

In [6]:
customer_frequency_query = """
WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(DISTINCT invoice) AS order_count
    FROM retail_transactions
    GROUP BY customer_id
)

SELECT
    COUNT(*) AS total_customers,
    COUNT(*) FILTER (WHERE order_count = 1) AS one_time_customers,
    COUNT(*) FILTER (WHERE order_count > 1) AS repeat_customers,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE order_count > 1)
        / COUNT(*),
        2
    ) AS repeat_customer_pct,
    ROUND(AVG(order_count), 2) AS avg_orders_per_customer,
    MAX(order_count) AS max_orders
FROM customer_orders;
"""

with engine.connect() as connection:
    customer_frequency = pd.read_sql(
        text(customer_frequency_query),
        connection
    )

customer_frequency

,total_customers,one_time_customers,repeat_customers,repeat_customer_pct,avg_orders_per_customer,max_orders
0,5861,1625,4236,72.27,6.25,376


#### Interpretation

Repeat purchasing is a major characteristic of the customer base.

- Of **5,861 customers**, **4,236 (72.27%)** made more than one merchandise purchase.
- **1,625 customers (27.73%)** purchased only once during the available dataset period.
- Customers made an average of **6.25 orders** each.
- The most active customer placed **376 orders**, indicating substantial variation in purchasing frequency across customers.

The high proportion of repeat customers makes retention and churn analysis particularly relevant for this retailer. However, these figures describe behavior across the full dataset and should not be interpreted as the churn rate used later in the time-based predictive model.

#### 1.3.2 First vs Repeat Purchase Share

Each customer's invoices are ordered chronologically to classify their first observed purchase and subsequent repeat purchases.

In [7]:
repeat_order_query = """
WITH invoice_level AS (
    SELECT
        customer_id,
        invoice,
        MIN(invoice_date) AS invoice_date
    FROM retail_transactions
    GROUP BY customer_id, invoice
),

ranked_orders AS (
    SELECT
        customer_id,
        invoice,
        invoice_date,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY invoice_date, invoice
        ) AS order_number
    FROM invoice_level
)

SELECT
    COUNT(*) AS total_orders,
    COUNT(*) FILTER (WHERE order_number = 1) AS first_orders,
    COUNT(*) FILTER (WHERE order_number > 1) AS repeat_orders,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE order_number > 1)
        / COUNT(*),
        2
    ) AS repeat_order_pct
FROM ranked_orders;
"""

with engine.connect() as connection:
    repeat_order_share = pd.read_sql(
        text(repeat_order_query),
        connection
    )

repeat_order_share

,total_orders,first_orders,repeat_orders,repeat_order_pct
0,36629,5861,30768,84.0


#### Interpretation

Repeat purchases account for the majority of order activity.

- The dataset contains **36,629 total merchandise orders**.
- **5,861 orders** are customers' first observed purchases.
- **30,768 orders** are repeat purchases.
- Therefore, **84.0% of all orders come from repeat customers after their first purchase**.

This shows that existing customers contribute heavily to transaction volume, which strengthens the business case for retention analysis.

### 1.4 Customer Revenue Concentration

Customer revenue concentration measures how dependent total sales are on the highest-value customers.

Customers are ranked by total merchandise revenue, and cumulative revenue share is calculated to determine how much revenue is generated by the top portions of the customer base.

#### 1.4.1 Revenue Contribution by Customer

In [8]:
customer_revenue_query = """
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(line_revenue) AS revenue
    FROM retail_transactions
    GROUP BY customer_id
),

ranked_customers AS (
    SELECT
        customer_id,
        revenue,
        ROW_NUMBER() OVER (
            ORDER BY revenue DESC
        ) AS customer_rank,
        SUM(revenue) OVER (
            ORDER BY revenue DESC
        ) AS cumulative_revenue,
        SUM(revenue) OVER () AS total_revenue,
        COUNT(*) OVER () AS total_customers
    FROM customer_revenue
)

SELECT
    customer_id,
    customer_rank,
    ROUND(revenue, 2) AS revenue,
    ROUND(
        100.0 * customer_rank / total_customers,
        2
    ) AS cumulative_customer_pct,
    ROUND(
        100.0 * cumulative_revenue / total_revenue,
        2
    ) AS cumulative_revenue_pct
FROM ranked_customers
ORDER BY customer_rank
LIMIT 20;
"""

with engine.connect() as connection:
    customer_revenue = pd.read_sql(
        text(customer_revenue_query),
        connection
    )

customer_revenue

,customer_id,customer_rank,revenue,cumulative_customer_pct,cumulative_revenue_pct
0,18102,1,580987.04,0.02,3.40
1,14646,2,526751.52,0.03,6.49
2,14156,3,303609.17,0.05,8.27
3,14911,4,272801.15,0.07,9.86
4,17450,5,244784.25,0.09,11.30
5,13694,6,195640.69,0.10,12.44
6,17511,7,172132.87,0.12,13.45
7,16446,8,168472.50,0.14,14.44
8,16684,9,147142.77,0.15,15.30
9,12415,10,144033.37,0.17,16.14


#### Interpretation

Revenue is highly concentrated among a small number of customers.

- The highest-value customer generated approximately **£580,987**, representing **3.40% of total merchandise revenue** alone.
- The top **5 customers** generated about **11.30%** of total revenue.
- The top **10 customers** generated about **16.14%**.
- The top **20 customers** represent only about **0.34% of the customer base**, yet generated approximately **21.98% of total merchandise revenue**.

This suggests that retaining high-value customers could have a disproportionately large effect on overall revenue.

#### 1.4.2 Revenue Share of Top Customer Segments

Customers are ranked by total revenue and grouped according to their position in the customer base to measure how much total revenue is generated by the highest-value customer segments.


In [9]:
revenue_concentration_query = """
WITH customer_revenue AS (
    SELECT
        customer_id,
        SUM(line_revenue) AS revenue
    FROM retail_transactions
    GROUP BY customer_id
),

ranked_customers AS (
    SELECT
        customer_id,
        revenue,
        ROW_NUMBER() OVER (
            ORDER BY revenue DESC
        ) AS customer_rank,
        COUNT(*) OVER () AS total_customers,
        SUM(revenue) OVER () AS total_revenue
    FROM customer_revenue
)

SELECT
    ROUND(
        100.0 * SUM(revenue) FILTER (
            WHERE customer_rank <= CEIL(total_customers * 0.01)
        ) / MAX(total_revenue),
        2
    ) AS top_1_pct_revenue,

    ROUND(
        100.0 * SUM(revenue) FILTER (
            WHERE customer_rank <= CEIL(total_customers * 0.05)
        ) / MAX(total_revenue),
        2
    ) AS top_5_pct_revenue,

    ROUND(
        100.0 * SUM(revenue) FILTER (
            WHERE customer_rank <= CEIL(total_customers * 0.10)
        ) / MAX(total_revenue),
        2
    ) AS top_10_pct_revenue,

    ROUND(
        100.0 * SUM(revenue) FILTER (
            WHERE customer_rank <= CEIL(total_customers * 0.20)
        ) / MAX(total_revenue),
        2
    ) AS top_20_pct_revenue

FROM ranked_customers;
"""

with engine.connect() as connection:
    revenue_concentration = pd.read_sql(
        text(revenue_concentration_query),
        connection
    )

revenue_concentration

,top_1_pct_revenue,top_5_pct_revenue,top_10_pct_revenue,top_20_pct_revenue
0,32.09,52.03,63.92,77.2


#### Interpretation

Revenue is highly concentrated among the retailer's highest-value customers.

- The top **1% of customers** generate **32.09%** of total merchandise revenue.
- The top **5%** generate **52.03%**.
- The top **10%** generate **63.92%**.
- The top **20%** generate **77.20%**.

This means that approximately one-fifth of the customer base generates more than three-quarters of merchandise revenue. Losing even a relatively small number of high-value customers could therefore create substantial financial risk.

These figures represent gross positive merchandise sales; cancellations and returns are not included in this revenue measure.


### 1.5 Customer Purchase Recency

Purchase recency measures how long it has been since each customer's most recent merchandise purchase.

Customers with longer periods of inactivity may represent greater retention risk.

#### 1.5.1 Recency Distribution

In [10]:
recency_query = """
WITH dataset_end AS (
    SELECT MAX(invoice_date)::date AS reference_date
    FROM retail_transactions
),

customer_last_purchase AS (
    SELECT
        customer_id,
        MAX(invoice_date)::date AS last_purchase_date
    FROM retail_transactions
    GROUP BY customer_id
),

customer_recency AS (
    SELECT
        c.customer_id,
        c.last_purchase_date,
        d.reference_date,
        d.reference_date - c.last_purchase_date AS recency_days
    FROM customer_last_purchase c
    CROSS JOIN dataset_end d
)

SELECT
    CASE
        WHEN recency_days <= 30 THEN '0-30 days'
        WHEN recency_days <= 60 THEN '31-60 days'
        WHEN recency_days <= 90 THEN '61-90 days'
        WHEN recency_days <= 180 THEN '91-180 days'
        ELSE '181+ days'
    END AS recency_group,
    COUNT(*) AS customers,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS customer_pct
FROM customer_recency
GROUP BY
    CASE
        WHEN recency_days <= 30 THEN '0-30 days'
        WHEN recency_days <= 60 THEN '31-60 days'
        WHEN recency_days <= 90 THEN '61-90 days'
        WHEN recency_days <= 180 THEN '91-180 days'
        ELSE '181+ days'
    END
ORDER BY MIN(recency_days);
"""

with engine.connect() as connection:
    recency_distribution = pd.read_sql(
        text(recency_query),
        connection
    )

recency_distribution

,recency_group,customers,customer_pct
0,0-30 days,1666,28.43
1,31-60 days,742,12.66
2,61-90 days,477,8.14
3,91-180 days,589,10.05
4,181+ days,2387,40.73


#### Interpretation

Customer recency shows a highly mixed level of engagement across the customer base.

- **28.43%** of customers purchased within the last 30 days.
- A further **20.80%** purchased between 31 and 90 days ago.
- **10.05%** have been inactive for 91–180 days.
- The largest group, **40.73% of customers**, has not made a merchandise purchase for more than 180 days.

This indicates that while a substantial active customer base remains, a large share of customers has experienced prolonged inactivity.

Recency alone is not treated as the final churn definition; later churn modeling will use the predefined 180-day observation and 90-day future label windows at historical cutoff dates.